# การแปลงค่าพิกัดระหว่างกรอบอ้างอิง ITRF แบบ Step by Step
**วิชา 2108415 Geodetic Survey — ปฏิบัติการที่ 10**
ภาควิชาวิศวกรรมสำรวจ จุฬาลงกรณ์มหาวิทยาลัย

### วัตถุประสงค์
1. เข้าใจการแปลง Helmert 14 พารามิเตอร์ (7 พารามิเตอร์ + อัตราการเปลี่ยนแปลงอีก 7 ค่า) ที่ใช้แปลงค่าระหว่าง ITRF
2. คำนวณการแปลงด้วยมือ (NumPy) ทีละขั้น แล้วตรวจสอบกับ `pyproj`
3. แยกให้ออกระหว่าง **การเปลี่ยนกรอบอ้างอิง (frame)** กับ **การเปลี่ยน epoch (plate motion)**

| แบบฝึกหัด | โจทย์ |
|---|---|
| **Ex1** | แปลงข้าม ITRF โดย epoch ต้นทาง = epoch ปลายทาง |
| **Ex2 แบบที่ 1** | epoch ต่างกัน: **แปลงกรอบก่อน** (ที่ epoch ต้นทาง) → แปลงความเร็ว → เลื่อนค่าพิกัดไปยัง epoch ปลายทาง |
| **Ex2 แบบที่ 2** | epoch ต่างกัน: **เลื่อนค่าพิกัดไปยัง epoch ปลายทางก่อน** (ในกรอบต้นทาง) → แปลงกรอบที่ epoch ปลายทาง |
| **ส่วนที่ 3** | ทำทั้งหมดด้วย `pyproj` (รวมถึงการทำทุกขั้นตอนใน PROJ pipeline เดียว) |

## 0. ทฤษฎี

### 0.1 การแปลง Helmert 7 พารามิเตอร์ (IERS convention, position vector)
$$
\begin{bmatrix}X\\Y\\Z\end{bmatrix}_{B}
=\begin{bmatrix}X\\Y\\Z\end{bmatrix}_{A}
+\begin{bmatrix}T_1\\T_2\\T_3\end{bmatrix}
+D\begin{bmatrix}X\\Y\\Z\end{bmatrix}_{A}
+\begin{bmatrix}0&-R_3&R_2\\R_3&0&-R_1\\-R_2&R_1&0\end{bmatrix}
\begin{bmatrix}X\\Y\\Z\end{bmatrix}_{A}
$$

### 0.2 พารามิเตอร์ขึ้นกับเวลา (14 พารามิเตอร์)
ITRF แต่ละรุ่นเคลื่อนที่สัมพัทธ์กันช้า ๆ พารามิเตอร์ทั้ง 7 ค่าจึงเป็นฟังก์ชันของเวลา
$$P(t) = P(t_0) + \dot P\,(t-t_0),\qquad P\in\{T_1,T_2,T_3,D,R_1,R_2,R_3\}$$
โดย $t_0$ คือ epoch อ้างอิงของชุดพารามิเตอร์ (เช่น 2010.0 สำหรับ ITRF2014↔ITRF2008) **ไม่ใช่** epoch ของค่าพิกัด

### 0.3 การแปลงความเร็ว (อนุพันธ์ของสมการ 0.1 เทียบกับเวลา ตัดพจน์อันดับสอง)
$$\dot{\mathbf X}_B = \dot{\mathbf X}_A + \dot{\mathbf T} + \dot D\,\mathbf X_A + \dot{\mathbf R}\,\mathbf X_A$$

### 0.4 การเลื่อน epoch ภายในกรอบเดียวกัน (plate motion, ความเร็วเชิงเส้น)
$$\mathbf X(t_2) = \mathbf X(t_1) + \dot{\mathbf X}\,(t_2-t_1)$$

### 0.5 หน่วยที่ใช้
| พารามิเตอร์ | ตาราง IGN/IERS | PROJ (`+proj=helmert`) | ตัวคูณ PROJ → IGN |
|---|---|---|---|
| $T$ | mm | m | ×1000 |
| $D$ | ppb ($10^{-9}$) | ppm ($10^{-6}$) | ×1000 |
| $R$ | mas (0.001″) | arc-second | ×1000 |
| อัตรา | ต่อปี | ต่อปี | ×1000 |

ที่มา: <https://itrf.ign.fr/docs/solutions/itrf2014/Transfo-ITRF2014_ITRFs.txt>

## 1. เตรียมเครื่องมือ

In [1]:
import numpy as np
import pandas as pd
import pyproj
from pyproj import Transformer, CRS
from pyproj.transformer import TransformerGroup

pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
print('pyproj', pyproj.__version__, '| PROJ', pyproj.proj_version_str)

# รหัส EPSG ของระบบพิกัดฉาก geocentric (ECEF) ของ ITRF แต่ละรุ่น
ITRF2020 = 'EPSG:9988'
ITRF2014 = 'EPSG:7789'
ITRF2008 = 'EPSG:5332'
ITRF2005 = 'EPSG:4896'
for c in (ITRF2020, ITRF2014, ITRF2008, ITRF2005):
    crs = CRS(c)
    print(f'{c:10s} {crs.name:10s} {crs.type_name}')

pyproj 3.7.2 | PROJ 9.5.1
EPSG:9988  ITRF2020   Geocentric CRS
EPSG:7789  ITRF2014   Geocentric CRS
EPSG:5332  ITRF2008   Geocentric CRS
EPSG:4896  ITRF2005   Geocentric CRS


## 2. ขั้นที่ 1 — อ่านพารามิเตอร์ 14 ค่าจาก PROJ

`Transformer.from_crs()` จะเลือก operation จากฐานข้อมูล EPSG แล้วสร้างนิยาม PROJ ให้ ลองดูว่าแปลง ITRF2014 → ITRF2008 ใช้พารามิเตอร์อะไร

In [2]:
t14_08 = Transformer.from_crs(ITRF2014, ITRF2008)
print('Description :', t14_08.description)
print('Definition  :', t14_08.definition)

Description : Inverse of ITRF2008 to ITRF2014 (1)
Definition  : proj=pipeline step inv proj=helmert x=-0.0016 y=-0.0019 z=-0.0024 rx=0 ry=0 rz=0 s=2e-05 dx=0 dy=0 dz=0.0001 drx=0 dry=0 drz=0 ds=-3e-05 t_epoch=2010 convention=position_vector


สังเกต:
* EPSG เก็บไว้เฉพาะ **ITRF2008 → ITRF2014** ดังนั้น PROJ จึงใช้ `step inv proj=helmert ...` (ใช้ฟังก์ชันผกผัน)
* หน่วยของ PROJ คือ m, ppm, arc-second (ดูตาราง 0.5)
* `convention=position_vector` ตรงกับสมการ 0.1

ต่อไปเขียนฟังก์ชันอ่านค่าเหล่านี้โดย **อ้างอิงจากชื่อพารามิเตอร์** (ไม่ใช่ตำแหน่งใน string) และ **ใส่เครื่องหมายลบให้เฉพาะกรณี `inv`**
> การกลับเครื่องหมายพารามิเตอร์ทุกตัวเป็นการประมาณอันดับหนึ่งของ Helmert ผกผัน ความคลาดเคลื่อนเกิดจากผลคูณของพารามิเตอร์เล็ก ๆ จึงน้อยกว่า 1 µm

In [3]:
def helmert_params(transformer):
    '''คืนค่า 14 พารามิเตอร์ในหน่วยของตาราง IGN: T (mm), D (ppb), R (mas) และอัตราต่อปี'''
    tokens = transformer.definition.split()
    if tokens.count('proj=helmert') != 1:
        raise ValueError('ฟังก์ชันนี้รองรับเฉพาะ operation ที่มี helmert 1 ขั้น: ' + transformer.definition)
    kv = dict(tok.split('=', 1) for tok in tokens if '=' in tok)
    sign = -1.0 if 'inv' in tokens else 1.0                          # inverse → กลับเครื่องหมาย
    rot = -1.0 if kv.get('convention') == 'coordinate_frame' else 1.0  # แปลงให้เป็น position vector
    g = lambda k: float(kv.get(k, 0.0)) * 1000.0 * sign + 0.0         # m→mm, ppm→ppb, ″→mas (+0.0 กัน -0.0)
    return dict(
        T  = np.array([g('x'),  g('y'),  g('z')]),
        D  = g('s'),
        R  = rot * np.array([g('rx'),  g('ry'),  g('rz')]),
        Td = np.array([g('dx'), g('dy'), g('dz')]),
        Dd = g('ds'),
        Rd = rot * np.array([g('drx'), g('dry'), g('drz')]),
        t0 = float(kv['t_epoch']),
        name = transformer.description,
    )

def show_params(p):
    return pd.DataFrame(
        {'T1 (mm)': [p['T'][0], p['Td'][0]], 'T2 (mm)': [p['T'][1], p['Td'][1]], 'T3 (mm)': [p['T'][2], p['Td'][2]],
         'D (ppb)': [p['D'], p['Dd']],
         'R1 (mas)': [p['R'][0], p['Rd'][0]], 'R2 (mas)': [p['R'][1], p['Rd'][1]], 'R3 (mas)': [p['R'][2], p['Rd'][2]]},
        index=[f"ค่าที่ t0 = {p['t0']}", 'อัตรา (/ปี)'])

p14_08 = helmert_params(t14_08)
print(p14_08['name'])
show_params(p14_08)

Inverse of ITRF2008 to ITRF2014 (1)


,T1 (mm),T2 (mm),T3 (mm),D (ppb),R1 (mas),R2 (mas),R3 (mas)
ค่าที่ t0 = 2010.0,1.6000,1.9000,2.4000,-0.0200,0.0000,0.0000,0.0000
อัตรา (/ปี),0.0000,0.0000,-0.1000,0.0300,0.0000,0.0000,0.0000


✅ ตรวจสอบกับตาราง IGN (ITRF2014 → ITRF2008, epoch 2010.0):
`T = 1.6, 1.9, 2.4 mm, D = −0.02 ppb, R = 0` และอัตรา `Ṫ = 0.0, 0.0, −0.1 mm/y, Ḋ = 0.03 ppb/y, Ṙ = 0` — ตรงกัน

ฟังก์ชันนี้ใช้ได้กับกรณีไม่มี `inv` ด้วย เช่น ITRF2014 → ITRF2020:

In [4]:
t14_20 = Transformer.from_crs(ITRF2014, ITRF2020)
print(t14_20.definition)
show_params(helmert_params(t14_20))

proj=helmert x=0.0014 y=0.0009 z=-0.0014 rx=0 ry=0 rz=0 s=0.00042 dx=0 dy=0.0001 dz=-0.0002 drx=0 dry=0 drz=0 ds=0 t_epoch=2015 ellps=GRS80


,T1 (mm),T2 (mm),T3 (mm),D (ppb),R1 (mas),R2 (mas),R3 (mas)
ค่าที่ t0 = 2015.0,1.4000,0.9000,-1.4000,0.4200,0.0000,0.0000,0.0000
อัตรา (/ปี),0.0000,0.1000,-0.2000,0.0000,0.0000,0.0000,0.0000


## 3. ขั้นที่ 2–3 — คำนวณพารามิเตอร์ที่ epoch $t$ แล้วแปลงค่าพิกัด

เขียนฟังก์ชันตามสมการ 0.1–0.4 โดยแสดงผลทุกขั้นเมื่อ `verbose=True`

In [5]:
MAS2RAD = np.radians(1e-3 / 3600.0)   # 1 mas → rad

def skew(r):
    '''เมทริกซ์การหมุนขนาดเล็ก (position vector) จาก r = [R1, R2, R3] (rad)'''
    return np.array([[    0, -r[2],  r[1]],
                     [ r[2],     0, -r[0]],
                     [-r[1],  r[0],     0]])

def params_at(p, t):
    '''สมการ 0.2: P(t) = P(t0) + Pdot (t - t0)'''
    dt = t - p['t0']
    return p['T'] + p['Td'] * dt, p['D'] + p['Dd'] * dt, p['R'] + p['Rd'] * dt

def helmert14(X, t, p, verbose=False):
    '''สมการ 0.1 ที่ epoch t; X หน่วย m'''
    X = np.asarray(X, dtype=float)
    T, D, R = params_at(p, t)
    T_m, D_s, R_rad = T * 1e-3, D * 1e-9, R * MAS2RAD
    dX = T_m + D_s * X + skew(R_rad) @ X
    if verbose:
        print(f'  epoch t = {t}, t - t0 = {t - p["t0"]:.2f} y')
        print(f'  T(t) = {np.round(T, 4)} mm,  D(t) = {D:.4f} ppb,  R(t) = {np.round(R, 4)} mas')
        print(f'  T             = {np.round(T_m*1e3, 4)} mm')
        print(f'  D·X           = {np.round(D_s*X*1e3, 4)} mm')
        print(f'  R·X           = {np.round(skew(R_rad) @ X*1e3, 4)} mm')
        print(f'  ΔX = รวม      = {np.round(dX*1e3, 4)} mm')
    return X + dX

def transform_velocity(V, X, p, verbose=False):
    '''สมการ 0.3; V หน่วย m/y'''
    V, X = np.asarray(V, dtype=float), np.asarray(X, dtype=float)
    dV = p['Td'] * 1e-3 + p['Dd'] * 1e-9 * X + skew(p['Rd'] * MAS2RAD) @ X
    if verbose:
        print(f'  Ṫ = {np.round(p["Td"], 4)} mm/y, Ḋ·X = {np.round(p["Dd"]*1e-9*X*1e3, 4)} mm/y, ΔV = {np.round(dV*1e3, 4)} mm/y')
    return V + dV

def propagate(X, V, t_from, t_to):
    '''สมการ 0.4: เลื่อนค่าพิกัดจาก t_from ไป t_to ในกรอบเดิม'''
    return np.asarray(X, dtype=float) + np.asarray(V, dtype=float) * (t_to - t_from)

def itrf_pyproj(X, src, dst, epoch):
    '''แปลงด้วย pyproj (4D: X, Y, Z, epoch)'''
    x, y, z, _ = Transformer.from_crs(src, dst).transform(*X, epoch)
    return np.array([x, y, z])

def compare(rows):
    '''ตารางเปรียบเทียบค่าพิกัด และผลต่างเทียบกับแถวแรก (mm)'''
    df = pd.DataFrame({k: v for k, v in rows.items()}, index=['X (m)', 'Y (m)', 'Z (m)']).T
    ref = df.iloc[0]
    for c in 'XYZ':
        df[f'Δ{c} (mm)'] = (df[f'{c} (m)'] - ref[f'{c} (m)']) * 1000
    return df

## 4. Ex1 — แปลงข้าม ITRF ที่ epoch ต้นทาง = epoch ปลายทาง

**โจทย์:** ค่าพิกัดในกรอบ ITRF2014 epoch 2020.0 ต้องการค่าพิกัดในกรอบ ITRF2008 **ที่ epoch เดียวกัน** (2020.0)

> หมายเหตุ: ไฟล์ `ITRF_Trans.py` เดิมกำหนด `f_epoch = 2015` ให้ค่าพิกัดชุดนี้ แต่ใน Ex2 แบบที่ 1 ค่าพิกัดชุดเดียวกันถูกใช้เป็นค่า **epoch 2020** (และเมื่อเลื่อนกลับไป 2015 จะตรงกับค่าอ้างอิง `xx, yy, zz`) ใน Notebook นี้จึงใช้ 2020.0 เพื่อให้ข้อมูลสอดคล้องกัน — เปลี่ยนค่า `epoch` เป็น 2015 ได้ ผลต่างระหว่างสองกรณีน้อยกว่า 1 mm เนื่องจากอัตราของ ITRF2014→ITRF2008 มีค่าน้อย

In [6]:
X14 = np.array([-1482252.181, 5925273.149, 1831476.151])   # ITRF2014 (m)
epoch = 2020.0

print('ขั้นที่ 1–3: แปลงกรอบด้วยมือ ITRF2014 → ITRF2008')
X08_manual = helmert14(X14, epoch, p14_08, verbose=True)

print('\nขั้นที่ 4: ตรวจสอบกับ pyproj')
X08_pyproj = itrf_pyproj(X14, ITRF2014, ITRF2008, epoch)

compare({'pyproj': X08_pyproj, 'คำนวณเอง (NumPy)': X08_manual, 'ค่าเดิม ITRF2014': X14})

ขั้นที่ 1–3: แปลงกรอบด้วยมือ ITRF2014 → ITRF2008
  epoch t = 2020.0, t - t0 = 10.00 y
  T(t) = [1.6 1.9 1.4] mm,  D(t) = 0.2800 ppb,  R(t) = [0. 0. 0.] mas
  T             = [1.6 1.9 1.4] mm
  D·X           = [-0.415   1.6591  0.5128] mm
  R·X           = [0. 0. 0.] mm
  ΔX = รวม      = [1.185  3.5591 1.9128] mm

ขั้นที่ 4: ตรวจสอบกับ pyproj


,X (m),Y (m),Z (m),ΔX (mm),ΔY (mm),ΔZ (mm)
pyproj,"-1,482,252.1798","5,925,273.1526","1,831,476.1529",0.0000,0.0000,0.0000
คำนวณเอง (NumPy),"-1,482,252.1798","5,925,273.1526","1,831,476.1529",0.0000,0.0000,0.0000
ค่าเดิม ITRF2014,"-1,482,252.1810","5,925,273.1490","1,831,476.1510",-1.1850,-3.5591,-1.9128


**สรุป Ex1:** การแปลง ITRF2014 → ITRF2008 ทำให้ค่าพิกัดเปลี่ยนในระดับ mm (เพราะสองกรอบเกือบเท่ากัน) และค่าที่คำนวณเองตรงกับ pyproj (ผลต่าง ≈ 0 mm)

## 5. Ex2 แบบที่ 1 — epoch ต่างกัน: แปลงกรอบ → แปลงความเร็ว → เลื่อน epoch

**โจทย์:** ค่าพิกัด ITRF2014 epoch **2020.0** และความเร็วของสถานี (ITRF2014) ต้องการค่าพิกัด ITRF2008 epoch **2015.0**

```
ITRF2014 @2020.0 ──(Helmert @2020.0)──► ITRF2008 @2020.0 ──(V_ITRF2008 × Δt)──► ITRF2008 @2015.0
```
ความเร็วต้องอยู่ในกรอบเดียวกับค่าพิกัด จึงต้องแปลงความเร็วจาก ITRF2014 เป็น ITRF2008 ก่อน (สมการ 0.3)

In [7]:
X14   = np.array([-1482252.181, 5925273.149, 1831476.151])   # ITRF2014 @2020.0 (m)
V14   = np.array([-0.02307, -0.00288, -0.00936])             # ITRF2014 (m/y)
f_epoch, t_epoch = 2020.0, 2015.0
X_ref = np.array([-1482252.0642, 5925273.1661, 1831476.1999]) # ค่าอ้างอิง ITRF2008 @2015.0 (xx, yy, zz)

print(f'ขั้นที่ 1: แปลงกรอบ ITRF2014 → ITRF2008 ที่ epoch ต้นทาง {f_epoch}')
X08_f = helmert14(X14, f_epoch, p14_08, verbose=True)

print('\nขั้นที่ 2: แปลงความเร็ว ITRF2014 → ITRF2008')
V08 = transform_velocity(V14, X14, p14_08, verbose=True)
print('  V(ITRF2014) =', V14, 'm/y')
print('  V(ITRF2008) =', np.round(V08, 6), 'm/y')

print(f'\nขั้นที่ 3: เลื่อน epoch ในกรอบ ITRF2008 จาก {f_epoch} → {t_epoch}  (Δt = {t_epoch - f_epoch} y)')
X08_t = propagate(X08_f, V08, f_epoch, t_epoch)
print('  V·Δt =', np.round(V08 * (t_epoch - f_epoch) * 1e3, 2), 'mm')

ex2_1 = compare({'ค่าอ้างอิง (xx,yy,zz)': X_ref, 'Ex2 แบบที่ 1': X08_t})
ex2_1

ขั้นที่ 1: แปลงกรอบ ITRF2014 → ITRF2008 ที่ epoch ต้นทาง 2020.0
  epoch t = 2020.0, t - t0 = 10.00 y
  T(t) = [1.6 1.9 1.4] mm,  D(t) = 0.2800 ppb,  R(t) = [0. 0. 0.] mas
  T             = [1.6 1.9 1.4] mm
  D·X           = [-0.415   1.6591  0.5128] mm
  R·X           = [0. 0. 0.] mm
  ΔX = รวม      = [1.185  3.5591 1.9128] mm

ขั้นที่ 2: แปลงความเร็ว ITRF2014 → ITRF2008
  Ṫ = [ 0.   0.  -0.1] mm/y, Ḋ·X = [-0.0445  0.1778  0.0549] mm/y, ΔV = [-0.0445  0.1778 -0.0451] mm/y
  V(ITRF2014) = [-0.02307 -0.00288 -0.00936] m/y
  V(ITRF2008) = [-0.023114 -0.002702 -0.009405] m/y

ขั้นที่ 3: เลื่อน epoch ในกรอบ ITRF2008 จาก 2020.0 → 2015.0  (Δt = -5.0 y)
  V·Δt = [115.57  13.51  47.03] mm


,X (m),Y (m),Z (m),ΔX (mm),ΔY (mm),ΔZ (mm)
"ค่าอ้างอิง (xx,yy,zz)","-1,482,252.0642","5,925,273.1661","1,831,476.1999",0.0000,0.0000,0.0000
Ex2 แบบที่ 1,"-1,482,252.0642","5,925,273.1661","1,831,476.1999",-0.0427,-0.0297,0.0381


**ตรวจสอบการแปลงความเร็วด้วย pyproj:** PROJ ไม่มีคำสั่งแปลงความเร็วโดยตรง แต่หาได้จากนิยาม
$\dot{\mathbf X}_B \approx \mathbf X_B(t+1) - \mathbf X_B(t)$ โดยที่ $\mathbf X_A(t+1) = \mathbf X_A(t) + \dot{\mathbf X}_A$

In [8]:
V08_pyproj = itrf_pyproj(X14 + V14, ITRF2014, ITRF2008, f_epoch + 1) - itrf_pyproj(X14, ITRF2014, ITRF2008, f_epoch)
pd.DataFrame({'NumPy (m/y)': V08, 'pyproj (m/y)': V08_pyproj, 'ผลต่าง (mm/y)': (V08 - V08_pyproj) * 1e3},
             index=['Vx', 'Vy', 'Vz']).style.format('{:.6f}')

,NumPy (m/y),pyproj (m/y),ผลต่าง (mm/y)
Vx,-0.023114,-0.023114,0.000000
Vy,-0.002702,-0.002702,-0.000000
Vz,-0.009405,-0.009405,0.000000


## 6. Ex2 แบบที่ 2 — epoch ต่างกัน: เลื่อน epoch → แปลงกรอบ

**โจทย์:** ค่าพิกัด ITRF2014 epoch **2020.14** และความเร็ว (ITRF2014) ต้องการค่าพิกัด ITRF2008 epoch **2013.81**

```
ITRF2014 @2020.14 ──(V_ITRF2014 × Δt)──► ITRF2014 @2013.81 ──(Helmert @2013.81)──► ITRF2008 @2013.81
```
ข้อดีของแบบนี้: ใช้ความเร็วในกรอบ ITRF2014 ได้เลย ไม่ต้องแปลงความเร็ว
**สำคัญ:** ขั้นที่ 2 ต้องคำนวณพารามิเตอร์ Helmert ที่ **epoch ปลายทาง** (2013.81) เพราะค่าพิกัดถูกเลื่อนมาอยู่ที่ epoch นั้นแล้ว

In [9]:
X14   = np.array([-1394062.2161, 5916172.6166, 1926911.2337])  # ITRF2014 @2020.14 (m)
V14   = np.array([-0.02327, -0.00251, -0.00914])               # ITRF2014 (m/y)
f_epoch, t_epoch = 2020.14, 2013.81

print(f'ขั้นที่ 1: เลื่อน epoch ในกรอบ ITRF2014 จาก {f_epoch} → {t_epoch}  (Δt = {t_epoch - f_epoch:.2f} y)')
X14_t = propagate(X14, V14, f_epoch, t_epoch)



print('  V·Δt =', np.round(V14 * (t_epoch - f_epoch) * 1e3, 2), 'mm')
print('  X(ITRF2014 @', t_epoch, ') =', np.round(X14_t, 4))

print(f'\nขั้นที่ 2: แปลงกรอบ ITRF2014 → ITRF2008 ที่ epoch ปลายทาง {t_epoch}')
X08_way2 = helmert14(X14_t, t_epoch, p14_08, verbose=True)

print('\nขั้นที่ 3: ตรวจสอบกับ pyproj')
X08_way2_pyproj = itrf_pyproj(X14_t, ITRF2014, ITRF2008, t_epoch)
compare({'แบบที่ 2 (pyproj)': X08_way2_pyproj, 'แบบที่ 2 (NumPy)': X08_way2})

ขั้นที่ 1: เลื่อน epoch ในกรอบ ITRF2014 จาก 2020.14 → 2013.81  (Δt = -6.33 y)
  V·Δt = [147.3   15.89  57.86] mm
  X(ITRF2014 @ 2013.81 ) = [-1394062.0688  5916172.6325  1926911.2916]

ขั้นที่ 2: แปลงกรอบ ITRF2014 → ITRF2008 ที่ epoch ปลายทาง 2013.81
  epoch t = 2013.81, t - t0 = 3.81 y
  T(t) = [1.6   1.9   2.019] mm,  D(t) = 0.0943 ppb,  R(t) = [0. 0. 0.] mas
  T             = [1.6   1.9   2.019] mm
  D·X           = [-0.1315  0.5579  0.1817] mm
  R·X           = [0. 0. 0.] mm
  ΔX = รวม      = [1.4685 2.4579 2.2007] mm

ขั้นที่ 3: ตรวจสอบกับ pyproj


,X (m),Y (m),Z (m),ΔX (mm),ΔY (mm),ΔZ (mm)
แบบที่ 2 (pyproj),"-1,394,062.0673","5,916,172.6349","1,926,911.2938",0.0000,0.0000,0.0000
แบบที่ 2 (NumPy),"-1,394,062.0673","5,916,172.6349","1,926,911.2938",0.0000,0.0000,0.0000


### 6.1 เปรียบเทียบแบบที่ 1 กับแบบที่ 2 (ข้อมูลชุดเดียวกัน)
ทั้งสองแบบควรให้ผลเท่ากันในระดับต่ำกว่า mm ลองคำนวณแบบที่ 1 ด้วยข้อมูลชุดนี้

In [10]:
X08_way1 = propagate(helmert14(X14, f_epoch, p14_08),
                     transform_velocity(V14, X14, p14_08), f_epoch, t_epoch)
compare({'แบบที่ 2: เลื่อน epoch → แปลงกรอบ': X08_way2,
         'แบบที่ 1: แปลงกรอบ → เลื่อน epoch': X08_way1})

,X (m),Y (m),Z (m),ΔX (mm),ΔY (mm),ΔZ (mm)
แบบที่ 2: เลื่อน epoch → แปลงกรอบ,"-1,394,062.0673","5,916,172.6349","1,926,911.2938",0.0000,0.0000,0.0000
แบบที่ 1: แปลงกรอบ → เลื่อน epoch,"-1,394,062.0673","5,916,172.6349","1,926,911.2938",0.0000,-0.0000,0.0000


**ข้อสังเกต:** ผลต่างระหว่างสองแบบมีค่าน้อยมาก (ระดับ µm) เกิดจากพจน์อันดับสองที่ตัดทิ้งในสมการแปลงความเร็ว

**ข้อผิดพลาดที่พบบ่อย:** เลื่อน epoch ก่อนแล้วแปลงกรอบที่ epoch **ต้นทาง** หรือใช้ความเร็วในกรอบ ITRF2014 กับค่าพิกัดในกรอบ ITRF2008 โดยไม่แปลง สำหรับ ITRF2014↔ITRF2008 ผลต่างยังน้อย แต่ระหว่างรุ่นที่ห่างกัน (เช่น ITRF2020 ↔ ITRF2000/ITRF97) อาจต่างกันหลาย mm ถึง cm

In [11]:
X08_wrong = helmert14(X14_t, f_epoch, p14_08)   # ❌ ใช้พารามิเตอร์ที่ epoch ต้นทาง
compare({'ถูกต้อง (Helmert @ t_epoch)': X08_way2, 'ผิด (Helmert @ f_epoch)': X08_wrong})

,X (m),Y (m),Z (m),ΔX (mm),ΔY (mm),ΔZ (mm)
ถูกต้อง (Helmert @ t_epoch),"-1,394,062.0673","5,916,172.6349","1,926,911.2938",0.0000,0.0000,0.0000
ผิด (Helmert @ f_epoch),"-1,394,062.0676","5,916,172.6361","1,926,911.2935",-0.2647,1.1235,-0.2671


## 7. ส่วนที่ 3 — แปลงข้าม ITRF ด้วย pyproj

### 7.1 ขั้นพื้นฐาน: `Transformer.from_crs()` แบบ 4 มิติ (X, Y, Z, epoch)
* ระบบ geocentric ทุกตัวรับค่าเป็น X, Y, Z (m) และ **ต้องส่ง epoch เป็นค่าตัวที่ 4** ถ้าไม่ส่ง PROJ จะไม่คิดส่วนของอัตรา (เหมือนใช้ $t = t_0$)
* pyproj **ไม่เลื่อน epoch ให้** ผลลัพธ์คือค่าพิกัดในกรอบใหม่ที่ epoch เดิม

In [12]:
X14   = np.array([-1394062.2161, 5916172.6166, 1926911.2337])
epoch = 2020.14

tr = Transformer.from_crs(ITRF2014, ITRF2008)
x, y, z, t = tr.transform(*X14, epoch)
print(f'มี epoch     : X = {x:.4f}  Y = {y:.4f}  Z = {z:.4f}  epoch = {t}')
x3, y3, z3 = tr.transform(*X14)            # ไม่ส่ง epoch
print(f'ไม่มี epoch  : X = {x3:.4f}  Y = {y3:.4f}  Z = {z3:.4f}')
print(f'ผลต่าง (mm)  : {(x-x3)*1e3:.3f}  {(y-y3)*1e3:.3f}  {(z-z3)*1e3:.3f}')

มี epoch     : X = -1394062.2149  Y = 5916172.6202  Z = 1926911.2356  epoch = 2020.14
ไม่มี epoch  : X = -1394062.2145  Y = 5916172.6184  Z = 1926911.2361
ผลต่าง (mm)  : -0.424  1.800  -0.428


### 7.2 ดู operation ที่มีให้เลือก (`TransformerGroup`) และการแปลงข้ามหลายรุ่น

In [13]:
for src, dst in [(ITRF2014, ITRF2008), (ITRF2020, ITRF2014), (ITRF2020, ITRF2005)]:
    grp = TransformerGroup(src, dst)
    print(f'{CRS(src).name} → {CRS(dst).name}')
    for tr in grp.transformers:
        print('   ', tr.description, '| accuracy =', tr.accuracy, 'm')
        print('   ', tr.definition)

# แปลงค่าพิกัด ITRF2014 → ITRF2020 → ITRF2005 ทีละขั้น เทียบกับแปลงตรง
X20 = itrf_pyproj(X14, ITRF2014, ITRF2020, epoch)
compare({'ITRF2014 → ITRF2005 (ตรง)': itrf_pyproj(X14, ITRF2014, ITRF2005, epoch),
         'ITRF2014 → ITRF2020 → ITRF2005': itrf_pyproj(X20, ITRF2020, ITRF2005, epoch)})

ITRF2014 → ITRF2008
    Inverse of ITRF2008 to ITRF2014 (1) | accuracy = 0.01 m
    proj=pipeline step inv proj=helmert x=-0.0016 y=-0.0019 z=-0.0024 rx=0 ry=0 rz=0 s=2e-05 dx=0 dy=0 dz=0.0001 drx=0 dry=0 drz=0 ds=-3e-05 t_epoch=2010 convention=position_vector
ITRF2020 → ITRF2014
    Inverse of ITRF2014 to ITRF2020 (1) | accuracy = 0.001 m
    proj=pipeline step inv proj=helmert x=0.0014 y=0.0009 z=-0.0014 rx=0 ry=0 rz=0 s=0.00042 dx=0 dy=0.0001 dz=-0.0002 drx=0 dry=0 drz=0 ds=0 t_epoch=2015 convention=position_vector
ITRF2020 → ITRF2005
    Inverse of ITRF2005 to ITRF2020 (1) | accuracy = 0.01 m
    proj=pipeline step inv proj=helmert x=-0.0027 y=-0.0001 z=0.0014 rx=0 ry=0 rz=0 s=-0.00065 dx=-0.0003 dy=0.0001 dz=-0.0001 drx=0 dry=0 drz=0 ds=-3e-05 t_epoch=2015 convention=position_vector


,X (m),Y (m),Z (m),ΔX (mm),ΔY (mm),ΔZ (mm)
ITRF2014 → ITRF2005 (ตรง),"-1,394,062.2122","5,916,172.6248","1,926,911.2327",0.0000,0.0000,0.0000
ITRF2014 → ITRF2020 → ITRF2005,"-1,394,062.2122","5,916,172.6248","1,926,911.2327",-0.0000,0.0000,0.0000


### 7.3 ผลลัพธ์เป็นละติจูด/ลองจิจูด/ความสูงทรงรี
ใช้ CRS แบบ Geographic 3D ของแต่ละ ITRF (ITRF2014 = EPSG:7912, ITRF2008 = EPSG:7911, ITRF2020 = EPSG:9989)
`always_xy=True` ทำให้ลำดับเป็น (lon, lat, h)

In [14]:
tr_geo = Transformer.from_crs(ITRF2014, 'EPSG:7911', always_xy=True)   # ITRF2014 XYZ → ITRF2008 lon/lat/h
lon, lat, h, _ = tr_geo.transform(*X14, epoch)
print(tr_geo.description)
print(f'lat = {lat:.9f}°, lon = {lon:.9f}°, h = {h:.4f} m')

Inverse of ITRF2008 to ITRF2014 (1) + Conversion from ITRF2008 (geocentric) to ITRF2008 (geog3D) + axis order change (2D)
lat = 17.700825365°, lon = 103.259069815°, h = 147.8729 m


### 7.4 ทำ Ex2 แบบที่ 2 ทั้งหมดใน PROJ pipeline เดียว
PROJ ไม่มีคำสั่ง "plate motion ด้วยความเร็วของสถานี" โดยตรง แต่ใช้ Helmert ที่มีเฉพาะ **อัตราการเลื่อนตำแหน่ง** แทนได้:
$$T(t) = \dot T\,(t - t_{0}),\quad \dot T = -\dot{\mathbf X},\ t_0 = t_{epoch}
\;\Rightarrow\; \mathbf X + T(t) = \mathbf X + \dot{\mathbf X}\,(t_{epoch} - t)$$
ซึ่งคือสมการ 0.4 พอดี จากนั้นใช้ `+proj=set +v_4=...` เปลี่ยนค่า epoch (ค่าพิกัดตัวที่ 4) ให้เป็น epoch ปลายทาง ก่อนเข้าขั้นแปลงกรอบ (ต้องใช้ PROJ ≥ 7.0)

```
+proj=pipeline
  +step +proj=helmert +dx=-Vx +dy=-Vy +dz=-Vz +t_epoch=t_epoch   ← plate motion
  +step +proj=set +v_4=t_epoch                                   ← epoch ใหม่
  +step <Helmert ITRF2014 → ITRF2008 จาก EPSG>                    ← แปลงกรอบ
```

In [15]:
def as_steps(definition):
    '''แปลง transformer.definition ให้เป็น "+step ..." เพื่อนำไปต่อใน pipeline'''
    toks = definition.split()
    toks = toks[1:] if toks[0] == 'proj=pipeline' else ['step'] + toks
    return ' '.join('+' + t for t in toks)

def itrf_with_epoch_pipeline(src, dst, V, f_epoch, t_epoch):
    vx, vy, vz = V
    pipe = ('+proj=pipeline '
            f'+step +proj=helmert +dx={-vx} +dy={-vy} +dz={-vz} +t_epoch={t_epoch} '
            f'+step +proj=set +v_4={t_epoch} '
            + as_steps(Transformer.from_crs(src, dst).definition))
    return Transformer.from_pipeline(pipe)

X14   = np.array([-1394062.2161, 5916172.6166, 1926911.2337])
V14   = np.array([-0.02327, -0.00251, -0.00914])
f_epoch, t_epoch = 2020.14, 2013.81

tp = itrf_with_epoch_pipeline(ITRF2014, ITRF2008, V14, f_epoch, t_epoch)
print(tp.definition, '\n')
x, y, z, t = tp.transform(*X14, f_epoch)
print(f'ผลลัพธ์ epoch = {t}')
compare({'pyproj pipeline เดียว': np.array([x, y, z]),
         'Ex2 แบบที่ 2 (NumPy)': X08_way2,
         'Ex2 แบบที่ 1 (NumPy)': X08_way1})

proj=pipeline step proj=helmert dx=0.02327 dy=0.00251 dz=0.00914 t_epoch=2013.81 step proj=set v_4=2013.81 step inv proj=helmert x=-0.0016 y=-0.0019 z=-0.0024 rx=0 ry=0 rz=0 s=2e-05 dx=0 dy=0 dz=0.0001 drx=0 dry=0 drz=0 ds=-3e-05 t_epoch=2010 convention=position_vector 

ผลลัพธ์ epoch = 2013.81


,X (m),Y (m),Z (m),ΔX (mm),ΔY (mm),ΔZ (mm)
pyproj pipeline เดียว,"-1,394,062.0673","5,916,172.6349","1,926,911.2938",0.0000,0.0000,0.0000
Ex2 แบบที่ 2 (NumPy),"-1,394,062.0673","5,916,172.6349","1,926,911.2938",0.0000,0.0000,0.0000
Ex2 แบบที่ 1 (NumPy),"-1,394,062.0673","5,916,172.6349","1,926,911.2938",0.0000,0.0000,0.0000


## 8. สรุป
| ขั้นตอน | สมการ | NumPy | pyproj |
|---|---|---|---|
| อ่านพารามิเตอร์ | — | `helmert_params()` | `Transformer.from_crs().definition` |
| พารามิเตอร์ที่ epoch $t$ | 0.2 | `params_at()` | อัตโนมัติเมื่อส่ง epoch ตัวที่ 4 |
| แปลงกรอบ | 0.1 | `helmert14()` | `transform(x, y, z, epoch)` |
| แปลงความเร็ว | 0.3 | `transform_velocity()` | ผลต่างของการแปลงที่ $t$ และ $t+1$ |
| เลื่อน epoch | 0.4 | `propagate()` | helmert ที่มีเฉพาะ `dx,dy,dz` + `proj=set` |

**ข้อควรจำ**
1. Epoch ของ **ค่าพิกัด** ต่างจาก epoch อ้างอิง $t_0$ ของ **พารามิเตอร์** เสมอ
2. การแปลงกรอบไม่เปลี่ยน epoch และการเลื่อน epoch ไม่เปลี่ยนกรอบ
3. ความเร็วกับค่าพิกัดต้องอยู่ในกรอบเดียวกัน
4. แปลงกรอบที่ epoch ที่ค่าพิกัดอยู่ ณ ขณะนั้น

## 9. แบบฝึกหัด
1. แปลงค่าพิกัดใน Ex2 แบบที่ 2 จาก ITRF2014 @2020.14 เป็น **ITRF2020** @2013.81 ทั้งด้วย NumPy และ pyproj
2. ทำข้อ 1 ด้วยแบบที่ 1 แล้วเปรียบเทียบผลต่างเป็น mm
3. แปลง ITRF2014 → ITRF2005 ที่ epoch 2000.0 และ 2025.0 แล้วอธิบายว่าผลต่างระหว่างสอง epoch มาจากพารามิเตอร์ใด

# ข้อ1 
แปลงค่าพิกัดใน Ex2 แบบที่ 2 จาก ITRF2014 @2020.14 เป็น **ITRF2020** @2013.81 ทั้งด้วย NumPy และ pyproj

In [16]:
t14_20 = Transformer.from_crs(ITRF2014, ITRF2020)
p14_20 = helmert_params(t14_20)
print(p14_20['name'])
show_params(p14_20)

ITRF2014 to ITRF2020 (1)


,T1 (mm),T2 (mm),T3 (mm),D (ppb),R1 (mas),R2 (mas),R3 (mas)
ค่าที่ t0 = 2015.0,1.4000,0.9000,-1.4000,0.4200,0.0000,0.0000,0.0000
อัตรา (/ปี),0.0000,0.1000,-0.2000,0.0000,0.0000,0.0000,0.0000


In [17]:
X14   = np.array([-1394062.2161, 5916172.6166, 1926911.2337])  # ITRF2014 @2020.14 (m)
V14   = np.array([-0.02327, -0.00251, -0.00914])               # ITRF2014 (m/y)
f_epoch, t_epoch = 2020.14, 2013.81

print(f'ขั้นที่ 1: เลื่อน epoch ในกรอบ ITRF2014 จาก {f_epoch} → {t_epoch}  (Δt = {t_epoch - f_epoch:.2f} y)')
X14_t = propagate(X14, V14, f_epoch, t_epoch)
print('  V·Δt =', np.round(V14 * (t_epoch - f_epoch) * 1e3, 2), 'mm')
print('  X(ITRF2014 @', t_epoch, ') =', np.round(X14_t, 4))

print(f'\nขั้นที่ 2: แปลงกรอบ ITRF2014 → ITRF2020 ที่ epoch ปลายทาง {t_epoch}')
X20_way2 = helmert14(X14_t, t_epoch, p14_20, verbose=True)

print('\nขั้นที่ 3: ตรวจสอบกับ pyproj')
X20_way2_pyproj = itrf_pyproj(X14_t, ITRF2014, ITRF2020, t_epoch)
compare({'แบบที่ 2 (pyproj)': X20_way2_pyproj, 'แบบที่ 2 (NumPy)': X20_way2})

ขั้นที่ 1: เลื่อน epoch ในกรอบ ITRF2014 จาก 2020.14 → 2013.81  (Δt = -6.33 y)
  V·Δt = [147.3   15.89  57.86] mm
  X(ITRF2014 @ 2013.81 ) = [-1394062.0688  5916172.6325  1926911.2916]

ขั้นที่ 2: แปลงกรอบ ITRF2014 → ITRF2020 ที่ epoch ปลายทาง 2013.81
  epoch t = 2013.81, t - t0 = -1.19 y
  T(t) = [ 1.4    0.781 -1.162] mm,  D(t) = 0.4200 ppb,  R(t) = [0. 0. 0.] mas
  T             = [ 1.4    0.781 -1.162] mm
  D·X           = [-0.5855  2.4848  0.8093] mm
  R·X           = [0. 0. 0.] mm
  ΔX = รวม      = [ 0.8145  3.2658 -0.3527] mm

ขั้นที่ 3: ตรวจสอบกับ pyproj


,X (m),Y (m),Z (m),ΔX (mm),ΔY (mm),ΔZ (mm)
แบบที่ 2 (pyproj),"-1,394,062.0680","5,916,172.6358","1,926,911.2912",0.0000,0.0000,0.0000
แบบที่ 2 (NumPy),"-1,394,062.0680","5,916,172.6358","1,926,911.2912",0.0000,0.0000,0.0000


# ข้อ2 
ทำข้อ 1 ด้วยแบบที่ 1 แล้วเปรียบเทียบผลต่างเป็น mm

In [18]:
print(f'ขั้นที่ 1: แปลงกรอบ ITRF2014 → ITRF2020 ที่ epoch ต้นทาง {f_epoch}')
X20_f = helmert14(X14, f_epoch, p14_20, verbose=True)

print(f'\nขั้นที่ 2: แปลงกรอบ ITRF2014 → ITRF2020 ที่ epoch ปลายทาง {t_epoch}')
V20 = transform_velocity(V14, X14, p14_20, verbose=True)
print('  V(ITRF2014) =', V14, 'm/y')
print('  V(ITRF2020) =', np.round(V20, 6), 'm/y')

print(f'\nขั้นที่ 3: เลื่อน epoch ในกรอบ ITRF2020 จาก {f_epoch} → {t_epoch}  (Δt = {t_epoch - f_epoch} y)')
X20_t = propagate(X20_f, V20, f_epoch, t_epoch)
print('  V·Δt =', np.round(V20 * (t_epoch - f_epoch) * 1e3, 2), 'mm')

ขั้นที่ 1: แปลงกรอบ ITRF2014 → ITRF2020 ที่ epoch ต้นทาง 2020.14
  epoch t = 2020.14, t - t0 = 5.14 y
  T(t) = [ 1.4    1.414 -2.428] mm,  D(t) = 0.4200 ppb,  R(t) = [0. 0. 0.] mas
  T             = [ 1.4    1.414 -2.428] mm
  D·X           = [-0.5855  2.4848  0.8093] mm
  R·X           = [0. 0. 0.] mm
  ΔX = รวม      = [ 0.8145  3.8988 -1.6187] mm

ขั้นที่ 2: แปลงกรอบ ITRF2014 → ITRF2020 ที่ epoch ปลายทาง 2013.81
  Ṫ = [ 0.   0.1 -0.2] mm/y, Ḋ·X = [-0.  0.  0.] mm/y, ΔV = [ 0.   0.1 -0.2] mm/y
  V(ITRF2014) = [-0.02327 -0.00251 -0.00914] m/y
  V(ITRF2020) = [-0.02327 -0.00241 -0.00934] m/y

ขั้นที่ 3: เลื่อน epoch ในกรอบ ITRF2020 จาก 2020.14 → 2013.81  (Δt = -6.330000000000155 y)
  V·Δt = [147.3   15.26  59.12] mm


In [19]:
compare({'แบบที่ 2: เลื่อน epoch → แปลงกรอบ': X20_way2,
         'แบบที่ 1: แปลงกรอบ → เลื่อน epoch': X20_t})

,X (m),Y (m),Z (m),ΔX (mm),ΔY (mm),ΔZ (mm)
แบบที่ 2: เลื่อน epoch → แปลงกรอบ,"-1,394,062.0680","5,916,172.6358","1,926,911.2912",0.0000,0.0000,0.0000
แบบที่ 1: แปลงกรอบ → เลื่อน epoch,"-1,394,062.0680","5,916,172.6358","1,926,911.2912",0.0000,0.0000,-0.0000


# ข้อ3
 แปลง ITRF2014 → ITRF2005 ที่ epoch 2000.0 และ 2025.0 แล้วอธิบายว่าผลต่างระหว่างสอง epoch มาจากพารามิเตอร์ใด

In [20]:
t14_05 = Transformer.from_crs(ITRF2014, ITRF2005)
p14_05 = helmert_params(t14_05)
print(p14_05['name'])
show_params(p14_05)

Inverse of ITRF2005 to ITRF2014 (1)


,T1 (mm),T2 (mm),T3 (mm),D (ppb),R1 (mas),R2 (mas),R3 (mas)
ค่าที่ t0 = 2010.0,2.6000,1.0000,-2.3000,0.9200,0.0000,0.0000,0.0000
อัตรา (/ปี),0.3000,0.0000,-0.1000,0.0300,0.0000,0.0000,0.0000


In [21]:
X14   = np.array([-1394062.2161, 5916172.6166, 1926911.2337])  # ITRF2014 @2020.14 (m)
V14   = np.array([-0.02327, -0.00251, -0.00914])               # ITRF2014 (m/y)
f_epoch, t_epoch = 2000.0, 2025.0

print(f'ขั้นที่ 1: แปลงกรอบ ITRF2014 → ITRF2005 ที่ epoch ต้นทาง {f_epoch}')
X05_f = helmert14(X14, f_epoch, p14_05, verbose=True) #ทรานส์ฟอร์มข้ามแกน

print('\nขั้นที่ 2: แปลงความเร็ว ITRF2014 → ITRF2005')
V05 = transform_velocity(V14, X14, p14_05, verbose=True) #ย้าย epoch
print('  V(ITRF2014) =', V14, 'm/y')
print('  V(ITRF2005) =', np.round(V05, 6), 'm/y')

print(f'\nขั้นที่ 3: เลื่อน epoch ในกรอบ ITRF2020 จาก {f_epoch} → {t_epoch}  (Δt = {t_epoch - f_epoch} y)')
X05_t = propagate(X05_f, V05, f_epoch, t_epoch)
print('  V·Δt =', np.round(V05 * (t_epoch - f_epoch) * 1e3, 2), 'mm')

com = compare({'ITRF 2005 @2000.0': X05_f, 'ITRF 2005 @2025.0': X05_t})
com


ขั้นที่ 1: แปลงกรอบ ITRF2014 → ITRF2005 ที่ epoch ต้นทาง 2000.0
  epoch t = 2000.0, t - t0 = -10.00 y
  T(t) = [-0.4  1.  -1.3] mm,  D(t) = 0.6200 ppb,  R(t) = [0. 0. 0.] mas
  T             = [-0.4  1.  -1.3] mm
  D·X           = [-0.8643  3.668   1.1947] mm
  R·X           = [0. 0. 0.] mm
  ΔX = รวม      = [-1.2643  4.668  -0.1053] mm

ขั้นที่ 2: แปลงความเร็ว ITRF2014 → ITRF2005
  Ṫ = [ 0.3  0.  -0.1] mm/y, Ḋ·X = [-0.0418  0.1775  0.0578] mm/y, ΔV = [ 0.2582  0.1775 -0.0422] mm/y
  V(ITRF2014) = [-0.02327 -0.00251 -0.00914] m/y
  V(ITRF2005) = [-0.023012 -0.002333 -0.009182] m/y

ขั้นที่ 3: เลื่อน epoch ในกรอบ ITRF2020 จาก 2000.0 → 2025.0  (Δt = 25.0 y)
  V·Δt = [-575.3   -58.31 -229.55] mm


,X (m),Y (m),Z (m),ΔX (mm),ΔY (mm),ΔZ (mm)
ITRF 2005 @2000.0,"-1,394,062.2174","5,916,172.6213","1,926,911.2336",0.0000,0.0000,0.0000
ITRF 2005 @2025.0,"-1,394,062.7927","5,916,172.5630","1,926,911.0040",-575.2955,-58.3129,-229.5548


ค่าความต่างเกิดจากพารามิเตอร์  V·Δt = [-575.3   -58.31 -229.55] ทำให้มีค่าต่างในแกน X Y Z ตามลำดับ

# สอบ

## 3


In [22]:
from datetime import datetime
import pyproj

def xyz2utm(x,y,z,zone):
    utm = "epsg:326" + zone # WGS84 UTM Zone 47N  
    ecef = {"proj":'geocent', "ellps":'WGS84'}
    transproj = pyproj.Transformer.from_crs(ecef,utm) # always_xy หมายถึงให้เรียง lon,lat
    E,N,h = transproj.transform(x,y,z)
    return E,N,h

def decimal_year(y, m, d):
    date = datetime(y, m, d)
    start = datetime(y, 1, 1)
    end = datetime(y + 1, 1, 1)

    year = y + (date - start).total_seconds() / (end - start).total_seconds()

    return year

def velocity_ITRF(x1, y1, z1, x2, y2, z2, epoch1, epoch2):
    vx = (x2 - x1) / (epoch2 - epoch1)
    vy = (y2 - y1) / (epoch2 - epoch1)
    vz = (z2 - z1) / (epoch2 - epoch1)

    return vx*1000, vy*1000, vz*1000

def velocity_ITRFUTM(x1, y1, z1, x2, y2, z2, epoch1, epoch2, zone):
    zone = str(zone)
    E1, N1, h1 = xyz2utm(x1, y1, z1, zone)
    E2, N2, h2 = xyz2utm(x2, y2, z2, zone)
    vE = (E2 - E1) / (epoch2 - epoch1)
    vN = (N2 - N1) / (epoch2 - epoch1)
    vh = (h2 - h1) / (epoch2 - epoch1)

    return vE*1000, vN*1000, vh*1000

def propagate_ITRF(x1, y1, z1, vx, vy, vz, epoch1, epoch2):
    x2 = x1 + vx * (epoch2 - epoch1)
    y2 = y1 + vy * (epoch2 - epoch1)
    z2 = z1 + vz * (epoch2 - epoch1)

    return x2, y2, z2

In [23]:
import numpy as np
from pyproj import Transformer

def transform_itrf_epoch(
    x, y, z,
    vx, vy, vz,
    f_ITRF, t_ITRF,
    f_epoch, t_epoch,
    params=None,
    method="manual",
    velocity_unit="mm/year",
    verbose=False
):
    """
    Transform coordinates between ITRF realizations and epochs.

    Input:
        x, y, z       : Initial ECEF coordinates (m)
        vx, vy, vz    : Initial velocity (mm/year or m/year)
        f_ITRF        : Source ITRF
        t_ITRF        : Target ITRF
        f_epoch       : Initial coordinate epoch
        t_epoch       : Target coordinate epoch
        params        : Helmert 14 parameters
        method        : 'manual' or 'pyproj'
        velocity_unit : 'mm/year' or 'm/year'

    Output:
        X_target      : Transformed coordinates (m)
    """

    # 1. Initial coordinates
    X0 = np.array([x, y, z], dtype=float)

    # 2. Convert velocity to m/year
    V0 = np.array([vx, vy, vz], dtype=float)

    if velocity_unit == "mm/year":
        V0 = V0 / 1000.0
    elif velocity_unit != "m/year":
        raise ValueError("velocity_unit must be mm/year or m/year")

    # 3. Coordinate propagation
    dt = t_epoch - f_epoch

    X_source = propagate(
        X0, V0,
        f_epoch, t_epoch
    )

    # 4. Reference frame transformation
    if f_ITRF == t_ITRF:
        X_target = X_source

    elif method == "manual":

        if params is None:
            raise ValueError("Helmert parameters are required")

        X_target = helmert14(
            X_source,
            t_epoch,
            params,
            verbose=verbose
        )

    elif method == "pyproj":

        X_target = itrf_pyproj(
            X_source,
            f_ITRF,
            t_ITRF,
            t_epoch
        )

    else:
        raise ValueError("method must be 'manual' or 'pyproj'")

    if verbose:
        print("\n===== ITRF Transformation Summary =====")
        print(f"Source ITRF : {f_ITRF}")
        print(f"Target ITRF : {t_ITRF}")
        print(f"Initial Epoch : {f_epoch}")
        print(f"Target Epoch  : {t_epoch}")
        print(f"Delta Epoch   : {dt:.10f} years")
        print(f"Initial XYZ   : {X0}")
        print(f"Propagated XYZ: {X_source}")
        print(f"Final XYZ     : {X_target}")

    return X_target


In [24]:
x1 = -1046898.3169
y1 = 5950358.6460
z1 = 2037915.1769
x2 = -1046898.4738
y2 = 5950358.6368
z2 = 2037915.1492

Y2021 = decimal_year(2021, 9, 7)
Y2025 = decimal_year(2025, 4, 8)
print(f'Y2021 = {Y2021:.4f}, Y2025 = {Y2025:.4f}')

vx, vy, vz = velocity_ITRF(x1, y1, z1, x2, y2, z2, Y2021, Y2025)
print(f'Velocity in ITRF: vx = {vx:.4f} mm/y, vy = {vy:.4f} mm/y, vz = {vz:.4f} mm/y')

vE, vN, vh = velocity_ITRFUTM(x1, y1, z1, x2, y2, z2, Y2021, Y2025, 47)
print(f'Velocity in UTM: vE = {vE:.4f} mm/y, vN = {vN:.4f} mm/y, vh = {vh:.4f} mm/y')

Y2021 = 2021.6822, Y2025 = 2025.2658
Velocity in ITRF: vx = -43.7833 mm/y, vy = -2.5673 mm/y, vz = -7.7297 mm/y
Velocity in UTM: vE = 43.6008 mm/y, vN = -8.7036 mm/y, vh = 2.3041 mm/y


### 3.1 เปลี่ยน Epoch

In [25]:
X2020 = [x1, y1, z1]  
V2020 = [vx/1000, vy/1000, vz/1000]  # Convert mm/y to m/y
f_epoch = Y2021
t_epoch = 2023.23


# เปลี่ยน epoch / ITRF เดิม
X2020_t = propagate_ITRF(X2020[0], X2020[1], X2020[2], V2020[0], V2020[1], V2020[2], f_epoch, t_epoch)
print(f'Propagated ITRF coordinates at {t_epoch:.2f}: X = {X2020_t[0]:.4f}, Y = {X2020_t[1]:.4f}, Z = {X2020_t[2]:.4f}')


Propagated ITRF coordinates at 2023.23: X = -1046898.3847, Y = 5950358.6420, Z = 2037915.1649


### 3.2 เปลี่ยน Epoch และ กรอบ

In [26]:
t20_08 = Transformer.from_crs(ITRF2020, ITRF2008)
p20_08 = helmert_params(t20_08)

X2021 = [x1, y1, z1]  
V2021 = [vx/1000, vy/1000, vz/1000]  # Convert mm/y to m/y
f_epoch = Y2021
t_epoch = 2025.00

X20_t = propagate(X2021,V2021,f_epoch,t_epoch)
X08_manual = helmert14(X20_t, t_epoch, p20_08, verbose=True)
X08_pyproj = itrf_pyproj(X20_t, ITRF2020, ITRF2008, t_epoch)

compare({'ด้วยมือ': X08_manual, 'pyproj': X08_pyproj})



  epoch t = 2025.0, t - t0 = 10.00 y
  T(t) = [0.2 0.  4.3] mm,  D(t) = 0.0100 ppb,  R(t) = [0. 0. 0.] mas
  T             = [0.2 0.  4.3] mm
  D·X           = [-0.0105  0.0595  0.0204] mm
  R·X           = [0. 0. 0.] mm
  ΔX = รวม      = [0.1895 0.0595 4.3204] mm


,X (m),Y (m),Z (m),ΔX (mm),ΔY (mm),ΔZ (mm)
ด้วยมือ,"-1,046,898.4620","5,950,358.6375","2,037,915.1556",0.0000,0.0000,0.0000
pyproj,"-1,046,898.4620","5,950,358.6375","2,037,915.1556",0.0000,0.0000,0.0000


### 3.3 เปลี่ยน กรอบ

In [27]:
X2020_t #จาก 3.1 ITRF 2020 epoch 2023.23
epoch = 2023.23

X08_manual = helmert14(X2020_t, epoch, p20_08, verbose=True)
X08_pyproj = itrf_pyproj(X2020_t, ITRF2020, ITRF2008, epoch)

compare({'ด้วยมือ': X08_manual, 'pyproj': X08_pyproj, 'ค่าเดิม ITRF2020': X2020_t})


  epoch t = 2023.23, t - t0 = 8.23 y
  T(t) = [0.2   0.177 4.123] mm,  D(t) = -0.0431 ppb,  R(t) = [0. 0. 0.] mas
  T             = [0.2   0.177 4.123] mm
  D·X           = [ 0.0451 -0.2565 -0.0878] mm
  R·X           = [0. 0. 0.] mm
  ΔX = รวม      = [ 0.2451 -0.0795  4.0352] mm


,X (m),Y (m),Z (m),ΔX (mm),ΔY (mm),ΔZ (mm)
ด้วยมือ,"-1,046,898.3844","5,950,358.6419","2,037,915.1690",0.0000,0.0000,0.0000
pyproj,"-1,046,898.3844","5,950,358.6419","2,037,915.1690",0.0000,0.0000,0.0000
ค่าเดิม ITRF2020,"-1,046,898.3847","5,950,358.6420","2,037,915.1649",-0.2451,0.0795,-4.0352


## 4


In [28]:
def molodensky_badekas(X, params, verbose=True):
    """
    Molodensky-Badekas 7-Parameter Transformation

    Input:
        X       : Source coordinates [X, Y, Z] (m)
        params  : Transformation parameters

    Parameters:
        dX, dY, dZ : Translation (m)
        rx, ry, rz : Rotation (arcsec)
        ds         : Scale (ppm)
        Xm, Ym, Zm : Pivot coordinates (m)

    Output:
        Xt, Yt, Zt : Target coordinates (m)
    """

    X = np.asarray(X, dtype=float)

    T = np.array([
        params['dX'],
        params['dY'],
        params['dZ']
    ], dtype=float)

    R = np.radians(
        np.array([
            params['rx'],
            params['ry'],
            params['rz']
        ], dtype=float) / 3600.0
    )

    rx, ry, rz = R

    ds = params['ds'] * 1e-6

    P = np.array([
        params['Xm'],
        params['Ym'],
        params['Zm']
    ], dtype=float)

    M = np.array([
        [1 + ds,  rz,      -ry],
        [-rz,     1 + ds,   rx],
        [ry,     -rx,       1 + ds]
    ])

    X_target = P + T + M @ (X - P)

    if verbose:
        print("===== Molodensky-Badekas Transformation =====")
        print("\nSource Coordinates (m):")
        print(X)

        print("\nPivot Coordinates (m):")
        print(P)

        print("\nTranslation (m):")
        print(T)

        print("\nRotation (arcsec):")
        print([params['rx'], params['ry'], params['rz']])

        print("\nScale (ppm):")
        print(params['ds'])

        print("\nRotation-Scale Matrix:")
        print(M)

        print("\nTarget Coordinates (m):")
        print(X_target)

    return X_target


def bursa_wolf(X, params, verbose=True):
    """
    Bursa-Wolf 7-Parameter Transformation

    Input:
        X : Source ECEF coordinates [X, Y, Z] (m)

        params:
            dX, dY, dZ : Translation (m)
            rx, ry, rz : Rotation (arcsec)
            ds         : Scale (ppm)

    Output:
        Xt, Yt, Zt : Target ECEF coordinates (m)
    """

    X = np.asarray(X, dtype=float)

    T = np.array([
        params['dX'],
        params['dY'],
        params['dZ']
    ], dtype=float)

    R = np.radians(
        np.array([
            params['rx'],
            params['ry'],
            params['rz']
        ], dtype=float) / 3600.0
    )

    rx, ry, rz = R

    ds = params['ds'] * 1e-6

    R_matrix = np.array([
        [1,   rz,  -ry],
        [-rz, 1,    rx],
        [ry, -rx,   1]
    ])

    Xt = T + (1 + ds) * (R_matrix @ X)

    if verbose:
        print("===== Bursa-Wolf Transformation =====")

        print("\nSource Coordinates (m):")
        print(X)

        print("\nTranslation (m):")
        print(T)

        print("\nRotation (arcsec):")
        print([params['rx'], params['ry'], params['rz']])

        print("\nScale (ppm):")
        print(params['ds'])

        print("\nRotation Matrix:")
        print(R_matrix)

        print("\nTarget Coordinates (m):")
        print(Xt)

    return Xt


In [33]:

def inverse_molodensky_badekas(X, params, verbose=True):
    """
    Inverse Molodensky-Badekas 7-Parameter Transformation

    Input:
        X       : Target coordinates [X, Y, Z] (m)
        params  : Transformation parameters

    Parameters:
        dX, dY, dZ : Translation (m)
        rx, ry, rz : Rotation (arcsec)
        ds         : Scale (ppm)
        Xm, Ym, Zm : Pivot coordinates (m)

    Output:
        Xs, Ys, Zs : Source coordinates (m)
    """

    X = np.asarray(X, dtype=float)

    T = np.array([
        params['dX'],
        params['dY'],
        params['dZ']
    ], dtype=float)

    R = np.radians(
        np.array([
            params['rx'],
            params['ry'],
            params['rz']
        ], dtype=float) / 3600.0
    )

    rx, ry, rz = R

    ds = params['ds'] * 1e-6

    P = np.array([
        params['Xm'],
        params['Ym'],
        params['Zm']
    ], dtype=float)

    M = np.array([
        [1 + ds,  rz,      -ry],
        [-rz,     1 + ds,   rx],
        [ry,     -rx,       1 + ds]
    ])

    X_source = P + np.linalg.solve(M, X - P - T)

    if verbose:
        print("===== Inverse Molodensky-Badekas Transformation =====")

        print("\nTarget Coordinates (m):")
        print(X)

        print("\nPivot Coordinates (m):")
        print(P)

        print("\nTranslation (m):")
        print(T)

        print("\nRotation (arcsec):")
        print([params['rx'], params['ry'], params['rz']])

        print("\nScale (ppm):")
        print(params['ds'])

        print("\nRotation-Scale Matrix:")
        print(M)

        print("\nSource Coordinates (m):")
        print(X_source)

    return X_source


def inverse_bursa_wolf(X, params, verbose=True):
    """
    Inverse Bursa-Wolf 7-Parameter Transformation

    Input:
        X : Target ECEF coordinates [X, Y, Z] (m)

        params:
            dX, dY, dZ : Translation (m)
            rx, ry, rz : Rotation (arcsec)
            ds         : Scale (ppm)

    Output:
        Xs, Ys, Zs : Source ECEF coordinates (m)
    """

    X = np.asarray(X, dtype=float)

    T = np.array([
        params['dX'],
        params['dY'],
        params['dZ']
    ], dtype=float)

    R = np.radians(
        np.array([
            params['rx'],
            params['ry'],
            params['rz']
        ], dtype=float) / 3600.0
    )

    rx, ry, rz = R

    ds = params['ds'] * 1e-6

    R_matrix = np.array([
        [1,   rz,  -ry],
        [-rz, 1,    rx],
        [ry, -rx,   1]
    ])

    X_source = np.linalg.solve(
        (1 + ds) * R_matrix,
        X - T
    )

    if verbose:
        print("===== Inverse Bursa-Wolf Transformation =====")

        print("\nTarget Coordinates (m):")
        print(X)

        print("\nTranslation (m):")
        print(T)

        print("\nRotation (arcsec):")
        print([params['rx'], params['ry'], params['rz']])

        print("\nScale (ppm):")
        print(params['ds'])

        print("\nRotation Matrix:")
        print(R_matrix)

        print("\nSource Coordinates (m):")
        print(X_source)

    return X_source

In [39]:
X_source = np.array([
    -1046898.4738,
    5950358.6368,
    2037915.1492
])

params = {

    # Translation (m)
    'dX': -0.40,
    'dY':  0.50,
    'dZ':  0.04,

    # Rotation (arcsec)
    'rx':  0.034,
    'ry':  0.0012,
    'rz':  0.06113,

    # Scale (ppm)
    'ds': 0.1069,

    # Pivot point (m)
    'Xm': -1198145.3260,
    'Ym': 6042214.1323,
    'Zm': 1592432.8612
}

X_target = inverse_molodensky_badekas(
    X_source,
    params,
    verbose=True
)

print("\nFinal Result:")
print(f"X = {X_target[0]:.4f} m")
print(f"Y = {X_target[1]:.4f} m")
print(f"Z = {X_target[2]:.4f} m")


===== Inverse Molodensky-Badekas Transformation =====

Target Coordinates (m):
[-1046898.4738  5950358.6368  2037915.1492]

Pivot Coordinates (m):
[-1198145.326   6042214.1323  1592432.8612]

Translation (m):
[-0.4   0.5   0.04]

Rotation (arcsec):
[0.034, 0.0012, 0.06113]

Scale (ppm):
0.1069

Rotation-Scale Matrix:
[[ 1.00000011e+00  2.96366603e-07 -5.81776417e-09]
 [-2.96366603e-07  1.00000011e+00  1.64836652e-07]
 [ 5.81776417e-09 -1.64836652e-07  1.00000011e+00]]

Source Coordinates (m):
[-1046898.06015357  5950358.11801226  2037915.0455568 ]

Final Result:
X = -1046898.0602 m
Y = 5950358.1180 m
Z = 2037915.0456 m


In [ ]:
X_target = np.array([
    -1046898.4738,
    5950358.6368,
    2037915.1492
])

dx, dy, dz = -0.40, 0.50, 0.04
rx, ry, rz = np.radians(np.array([0.034, 0.0012, 0.06113]) / 3600)
ds = 0.1069 * 1e-6

P = np.array([
    -1198145.3260,
    6042214.1323,
    1592432.8612
])

T = np.array([dx, dy, dz])

M = np.array([
    [1 + ds,  rz,      -ry],
    [-rz,     1 + ds,   rx],
    [ry,     -rx,       1 + ds]
])

X_source = P + np.linalg.solve(M, X_target - P - T)

print("Source Coordinates (m):")
print("Xs =", X_source[0])
print("Ys =", X_source[1])
print("Zs =", X_source[2])


Source Coordinates (m):
Xs = -1046898.0601535675
Ys = 5950358.118012255
Zs = 2037915.0455567958
